# Modélisation : popularité et content-based

Split par utilisateur

- le dernier clic de chaque utilisateur sert de test, le reste de train
- évaluation sur 5 000 utilisateurs tirés au hasard
- métriques : on fait 5 recommandations, l'article réellement lu est-il dans les 5 recos et à quel rang ?
- on ne recommande pas un article déjà lu

In [1]:
import json
import sys
import warnings

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA

sys.path.insert(0, "../src")
from utils import N_RECO, SEED, hit_rate_at_k, load_clicks, load_embeddings, mrr_at_k, sample_eval_users, temporal_split

warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

## Chargement et split

In [2]:
clicks = load_clicks("../data")
train, test = temporal_split(clicks)
users_eval = sample_eval_users(test, n=5000, seed=SEED)

test_items = test[test["user_id"].isin(users_eval)].set_index("user_id")["click_article_id"].to_dict()
train_eval = train[train["user_id"].isin(users_eval)].sort_values("click_timestamp")
articles_vus = train_eval.groupby("user_id")["click_article_id"].apply(set).to_dict()
dernier_lu = train_eval.groupby("user_id")["click_article_id"].last().to_dict()

print("Clics train :", len(train))
print("Clics test :", len(test))
print("Utilisateurs évalués :", len(users_eval))

Clics train : 2665284
Clics test : 322897
Utilisateurs évalués : 5000


## 1. Baseline popularité

In [3]:
populaires = train["click_article_id"].value_counts().index

recos_pop = {}
for u in users_eval:
    recos_pop[u] = [a for a in populaires[:50] if a not in articles_vus[u]][:N_RECO]

metriques_pop = {"hit_rate_at_5": hit_rate_at_k(recos_pop, test_items), "mrr_at_5": mrr_at_k(recos_pop, test_items)}
print(metriques_pop)

{'hit_rate_at_5': 0.0348, 'mrr_at_5': 0.015740000000000007}


## 2. Content-based (embeddings 250 dimensions)

Profil utilisateur = moyenne des embeddings des articles lus, puis on prend les 5 articles les plus proches (similarité cosinus). Les scores sont calculés par paquets de 500 utilisateurs pour ne pas saturer la mémoire.

In [4]:
def content_based(emb, profil="moyenne"):
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    if profil == "moyenne":
        profils = np.array([emb[list(articles_vus[u])].mean(axis=0) for u in users_eval])
    else:
        profils = emb[[dernier_lu[u] for u in users_eval]]
    recos = {}
    for i in range(0, len(users_eval), 500):
        scores = profils[i:i + 500] @ emb.T
        for j, u in enumerate(users_eval[i:i + 500]):
            scores[j, list(articles_vus[u])] = -np.inf
            recos[u] = np.argsort(scores[j])[::-1][:N_RECO].tolist()
    return {"hit_rate_at_5": hit_rate_at_k(recos, test_items), "mrr_at_5": mrr_at_k(recos, test_items)}


embeddings = load_embeddings("../data")
metriques_cb250 = content_based(embeddings)
print(metriques_cb250)

{'hit_rate_at_5': 0.0088, 'mrr_at_5': 0.004803333333333332}


## 3. Variante : profil = dernier article lu

In [5]:
metriques_cb_dernier = content_based(embeddings, profil="dernier")
print(metriques_cb_dernier)

{'hit_rate_at_5': 0.013, 'mrr_at_5': 0.006983333333333333}


## 4. Réduction à 50 dimensions (PCA)

In [6]:
pca = PCA(n_components=50, random_state=SEED)
emb_50 = pca.fit_transform(embeddings).astype(np.float32)

print(f"Variance conservée : {pca.explained_variance_ratio_.sum():.1%}")
print(f"Taille : {embeddings.nbytes / 1e6:.0f} Mo -> {emb_50.nbytes / 1e6:.0f} Mo")

metriques_cb50 = content_based(emb_50)
print(metriques_cb50)

Variance conservée : 94.5%
Taille : 364 Mo -> 73 Mo


{'hit_rate_at_5': 0.0068, 'mrr_at_5': 0.003983333333333333}


## Comparaison

In [7]:
metriques = {
    "popularite": metriques_pop,
    "content_based_250": metriques_cb250,
    "content_based_dernier": metriques_cb_dernier,
    "content_based_pca50": metriques_cb50,
}
with open("../results/metrics_02.json", "w", encoding="utf-8") as f:
    json.dump(metriques, f, indent=2)

pd.DataFrame(metriques).T.round(4)

,hit_rate_at_5,mrr_at_5
popularite,0.0348,0.0157
content_based_250,0.0088,0.0048
content_based_dernier,0.0130,0.0070
content_based_pca50,0.0068,0.0040


## Conclusion

- La popularité bat le content-based : sur un site d'actu, tout le monde lit les mêmes articles du moment.
- Le dernier article lu bat le profil moyen (hit rate 0,0130 contre 0,0088) : le prochain clic dépend surtout de ce qu'on vient de lire, la moyenne dilue les sujets. Le modèle final (notebook 04) exploite ce signal par la co-visitation depuis le dernier article lu et garde le profil moyen pour le contenu.
- Le content-based reste utile : il personnalise et peut recommander un article tout neuf (il suffit de son embedding).
- La PCA à 50 dimensions divise la taille par 5 (de 364 Mo à 73 Mo) pour une perte limitée (hit rate de 0,0088 à 0,0068) : c'est cette version qu'on déploie.
- Sur le protocole sans fuite du notebook 04, le content-based limité aux articles de moins de 24 h fait 0,0622 et sert de brique dans l'hybride retenu : la fraîcheur compte plus que la similarité.